# Laboratorio 2: Preprocesamiento de datos

Armamos un dataset con los tres problemas que vimos que son comunes: datos faltantes, columnas categóricas y columnas en escalas distintas. Vamos a ir arreglando cada cosa como se haría antes de mandarle esto a un modelo.

In [1]:
import numpy as np
import pandas as pd

autos = pd.DataFrame({
    "marca": ["toyota", "ford", "toyota", "fiat", "ford", "fiat", "toyota", "ford"],
    "anio": [2015, 2018, 2020, np.nan, 2012, 2019, 2016, np.nan],
    "kilometraje": [80000, 45000, 12000, 95000, np.nan, 30000, 70000, 60000],
    "precio": [8500000, 12300000, 15800000, 6200000, 7100000, 13900000, 9200000, 10500000],
})

autos

,marca,anio,kilometraje,precio
0,toyota,2015.0,80000.0,8500000
1,ford,2018.0,45000.0,12300000
2,toyota,2020.0,12000.0,15800000
3,fiat,NaN,95000.0,6200000
4,ford,2012.0,NaN,7100000
5,fiat,2019.0,30000.0,13900000
6,toyota,2016.0,70000.0,9200000
7,ford,NaN,60000.0,10500000


## Datos faltantes

Primer paso: saber dónde faltan datos y cuántos.

In [2]:
autos.isna().sum()

marca          0
anio           2
kilometraje    1
precio         0
dtype: int64

Dos autos sin año, uno sin kilometraje. Con 8 filas nomás, borrarlas sería tirar un cuarto del dataset a la basura. Mejor completarlos.

In [3]:
autos["anio"] = autos["anio"].fillna(autos["anio"].median())
autos["kilometraje"] = autos["kilometraje"].fillna(autos["kilometraje"].median())

autos.isna().sum()

marca          0
anio           0
kilometraje    0
precio         0
dtype: int64

Usamos la mediana en vez de la media porque con pocos datos un outlier hace variar mucho a la media.

## Variable categórica

`marca` es texto. Un modelo no puede hacer cuentas con `"toyota"`, así que la convertimos a columnas de 0 y 1.

In [4]:
autos_codificado = pd.get_dummies(autos, columns=["marca"])
autos_codificado

,anio,kilometraje,precio,marca_fiat,marca_ford,marca_toyota
0,2015.0,80000.0,8500000,False,False,True
1,2018.0,45000.0,12300000,False,True,False
2,2020.0,12000.0,15800000,False,False,True
3,2017.0,95000.0,6200000,True,False,False
4,2012.0,60000.0,7100000,False,True,False
5,2019.0,30000.0,13900000,True,False,False
6,2016.0,70000.0,9200000,False,False,True
7,2017.0,60000.0,10500000,False,True,False


Ahora cada auto tiene un True en la columna de su marca y False en las otras dos. Perdimos el campo de texto y en su lugar tenemos algo que un modelo puede usar.

## Escalado

Miren la diferencia de magnitud entre `anio` (miles) y `precio` (millones):

In [5]:
autos_codificado[["anio", "kilometraje", "precio"]].describe()

,anio,kilometraje,precio
count,8.000000,8.000000,8.000000e+00
mean,2016.750000,56500.000000,1.043750e+07
std,2.492847,26875.373752,3.350879e+06
min,2012.000000,12000.000000,6.200000e+06
25%,2015.750000,41250.000000,8.150000e+06
50%,2017.000000,60000.000000,9.850000e+06
75%,2018.250000,72500.000000,1.270000e+07
max,2020.000000,95000.000000,1.580000e+07


In [6]:
from sklearn.preprocessing import StandardScaler

columnas_numericas = ["anio", "kilometraje", "precio"]

scaler = StandardScaler()
autos_codificado[columnas_numericas] = scaler.fit_transform(autos_codificado[columnas_numericas])

autos_codificado[columnas_numericas].describe()

,anio,kilometraje,precio
count,8.000000,8.000000,8.000000e+00
mean,0.000000,0.000000,1.387779e-17
std,1.069045,1.069045,1.069045e+00
min,-2.037014,-1.770115,-1.351907e+00
25%,-0.428845,-0.606612,-7.297906e-01
50%,0.107211,0.139223,-1.874326e-01
75%,0.643268,0.636446,7.218148e-01
max,1.393746,1.531448,1.710821e+00


Fíjense la media (prácticamente 0) y el desvío (prácticamente 1) de las tres columnas ahora. Son valores muchisimo mas estables.

## Train / test split

Para esto volvemos al dataset de Iris, ya bien armado y pensado para clasificar. Separamos antes de entrenar.

In [1]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

print("filas totales:", len(X))
print("filas de entrenamiento:", len(X_train))
print("filas de evaluacion:", len(X_test))

filas totales: 150
filas de entrenamiento: 105
filas de evaluacion: 45


70/30, como un buen fernet. `X_test` e `y_test` quedan guardados sin tocar hasta que tengamos un modelo entrenado y queramos ver qué tan bien predice con datos que nunca vio.

## Ejercicios

1. En vez de completar los faltantes de `anio` y `kilometraje` con la mediana, probar completarlos con la media y compará los valores que quedan. ¿Cambia mucho?

2. Agregar una columna `combustible` a `autos` (por ejemplo `["nafta", "diesel", "nafta", "nafta", "diesel", "nafta", "diesel", "nafta"]`) y aplicar `pd.get_dummies` incluyendo esa columna también.

3. Probar `MinMaxScaler` en lugar de `StandardScaler` sobre las mismas tres columnas numéricas (`from sklearn.preprocessing import MinMaxScaler`). Observar el resultado de `describe()`: ¿en qué rango quedan los valores ahora?

4. Repetir el `train_test_split` de Iris pero con `test_size=0.5`. ¿Cuántas filas quedan en cada parte? ¿Es útil ese split?

In [ ]:
# ejercicio 1

import numpy as np
import pandas as pd

autos = pd.DataFrame({
    "marca": ["toyota", "ford", "toyota", "fiat", "ford", "fiat", "toyota", "ford"],
    "anio": [2015, 2018, 2020, np.nan, 2012, 2019, 2016, np.nan],
    "kilometraje": [80000, 45000, 12000, 95000, np.nan, 30000, 70000, 60000],
    "precio": [8500000, 12300000, 15800000, 6200000, 7100000, 13900000, 9200000, 10500000],
})

autosmedia = autos.copy()
autosmediana = autos.copy()

autosmedia["anio"] = autos["anio"].fillna(autos["anio"].mean())
autosmedia["kilometraje"] = autos["kilometraje"].fillna(autos["kilometraje"].mean())

print(autosmedia)

autosmediana["anio"] = autos["anio"].fillna(autos["anio"].median())
autosmediana["kilometraje"] = autos["kilometraje"].fillna(autos["kilometraje"].median())

print(autosmediana)

#en la dataframe completada con la media, los valores se completan como flotantes.
#en la dataframe completada con la mediana, los valores se completan como enteros redondeados para arriba.

    marca         anio  kilometraje    precio
0  toyota  2015.000000      80000.0   8500000
1    ford  2018.000000      45000.0  12300000
2  toyota  2020.000000      12000.0  15800000
3    fiat  2016.666667      95000.0   6200000
4    ford  2012.000000      56000.0   7100000
5    fiat  2019.000000      30000.0  13900000
6  toyota  2016.000000      70000.0   9200000
7    ford  2016.666667      60000.0  10500000
    marca    anio  kilometraje    precio
0  toyota  2015.0      80000.0   8500000
1    ford  2018.0      45000.0  12300000
2  toyota  2020.0      12000.0  15800000
3    fiat  2017.0      95000.0   6200000
4    ford  2012.0      60000.0   7100000
5    fiat  2019.0      30000.0  13900000
6  toyota  2016.0      70000.0   9200000
7    ford  2017.0      60000.0  10500000


In [15]:
# ejercicio 2

import numpy as np
import pandas as pd

autos = pd.DataFrame({
    "marca": ["toyota", "ford", "toyota", "fiat", "ford", "fiat", "toyota", "ford"],
    "anio": [2015, 2018, 2020, np.nan, 2012, 2019, 2016, np.nan],
    "kilometraje": [80000, 45000, 12000, 95000, np.nan, 30000, 70000, 60000],
    "precio": [8500000, 12300000, 15800000, 6200000, 7100000, 13900000, 9200000, 10500000],
    "combustible": ["nafta", "diesel", "nafta", "nafta", "diesel", "nafta", "diesel", "nafta"],
})

autos_codificado = pd.get_dummies(autos, columns=["marca", "combustible"])
autos_codificado


,anio,kilometraje,precio,marca_fiat,marca_ford,marca_toyota,combustible_diesel,combustible_nafta
0,2015.0,80000.0,8500000,False,False,True,False,True
1,2018.0,45000.0,12300000,False,True,False,True,False
2,2020.0,12000.0,15800000,False,False,True,False,True
3,NaN,95000.0,6200000,True,False,False,False,True
4,2012.0,NaN,7100000,False,True,False,True,False
5,2019.0,30000.0,13900000,True,False,False,False,True
6,2016.0,70000.0,9200000,False,False,True,True,False
7,NaN,60000.0,10500000,False,True,False,False,True


In [ ]:
# ejercicio 3

from sklearn.preprocessing import MinMaxScaler

columnas_numericas = ["anio", "kilometraje", "precio"]

scaler = StandardScaler()
autos_codificado[columnas_numericas] = scaler.fit_transform(autos_codificado[columnas_numericas])

autos_codificado[columnas_numericas].describe()

#quedan valores muchísimo más específicos y mucho más pequeños, 
#con una media de 0 y una desviación estándar de 1. 

,anio,kilometraje,precio
count,6.000000e+00,7.000000e+00,8.000000e+00
mean,-2.819966e-14,7.930164e-18,1.387779e-17
std,1.095445e+00,1.080123e+00,1.069045e+00
min,-1.736486e+00,-1.639458e+00,-1.351907e+00
25%,-5.271476e-01,-6.893175e-01,-7.297906e-01
50%,1.240347e-01,1.490416e-01,-1.874326e-01
75%,7.752171e-01,7.079477e-01,7.218148e-01
max,1.240347e+00,1.453156e+00,1.710821e+00


In [ ]:
# ejercicio 4

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=0)

print("filas totales:", len(X))
print("filas de entrenamiento:", len(X_train))
print("filas de evaluacion:", len(X_test))


#quedan 75 filas de entrenamiento y 75 filas de evaluación, la información siendo repartida 
#equitativamente entre ambos.
#creo que no es muy útil este split, ya que no es necesario tener tantas filas
#de evaluación, y se pierde información que podría ser útil para entrenar el modelo.

filas totales: 150
filas de entrenamiento: 75
filas de evaluacion: 75
